# El Lado Oscuro de las Pruebas E2E

## Introducción

Las pruebas E2E son poderosas pero tienen limitaciones importantes. En este notebook exploramos los desafíos y antipatrones comunes.

---

## 1. Flaky Tests: El enemigo silencioso

Los **flaky tests** son pruebas que a veces pasan y a veces fallan sin cambios en el código. Son especialmente comunes en E2E por varias razones.

### Causas comunes de flaky tests en E2E

#### 1. Timing issues
```python
# ❌ MAL: Espera fija (frágil)
page.click('button')
time.sleep(2)  # ¿Qué si la red es lenta?
expect(page.locator('.result')).to_be_visible()

# ✅ BIEN: Espera inteligente (robusto)
page.click('button')
expect(page.locator('.result')).to_be_visible(timeout=5000)
```

#### 2. Dependencia del estado
```python
# ❌ MAL: Depende del estado de tests anteriores
def test_delete_task(page):
    # Asume que hay una tarea creada por otro test
    page.click('[data-testid="delete-button"]')
    expect(page.locator('[data-testid="task-item"]')).to_have_count(0)

# ✅ BIEN: Cada test es independiente
def test_delete_task(page):
    # Crea su propia tarea
    page.fill('[data-testid="task-title-input"]', 'Test')
    page.click('[data-testid="create-task-button"]')
    # Luego la elimina
    page.click('[data-testid="delete-button"]')
    expect(page.locator('[data-testid="task-item"]')).to_have_count(0)
```

#### 3. Dependencia de recursos externos
```python
# ❌ MAL: Depende de API externa que puede fallar
def test_external_api(page):
    page.goto('https://api-externa.com/data')
    expect(page.locator('.data')).to_be_visible()

# ✅ BIEN: Mock de API externa
def test_external_api(page):
    # Usar mock o stub de la API
    page.route('https://api-externa.com/data', lambda route: route.fulfill(
        status=200,
        body='{"data": "mocked"}'
    ))
    page.goto('https://api-externa.com/data')
    expect(page.locator('.data')).to_be_visible()
```

#### 4. Problemas de concurrencia
```python
# ❌ MAL: Múltiples tests escriben al mismo archivo
# Test 1
page.fill('[data-testid="task-title-input"]', 'Task 1')
page.click('[data-testid="create-task-button"]')

# Test 2 (ejecutado en paralelo)
page.fill('[data-testid="task-title-input"]', 'Task 2')
page.click('[data-testid="create-task-button"]')
# ¡Puede causar race conditions en el archivo JSON!

# ✅ BIEN: Cada test usa su propia base de datos
# Usar fixtures que crean bases de datos aisladas
@pytest.fixture
def isolated_database():
    db_file = f'test_{uuid.uuid4()}.json'
    yield db_file
    os.remove(db_file)
```

## 2. Aislamiento entre tests

El aislamiento es crucial para pruebas E2E confiables.

### Estrategias de aislamiento

#### 1. Limpieza antes de cada test
```python
@pytest.fixture(autouse=True)
def reset_database():
    """Limpia la base de datos antes de cada test."""
    # Eliminar archivo de datos
    if os.path.exists('data/tasks.json'):
        os.remove('data/tasks.json')
    yield
    # Limpieza adicional después del test
    if os.path.exists('data/tasks.json'):
        os.remove('data/tasks.json')
```

#### 2. Bases de datos en memoria
```python
@pytest.fixture
def in_memory_db():
    """Usa una base de datos en memoria para cada test."""
    db = InMemoryDatabase()
    yield db
    # Se destruye automáticamente al terminar
```

#### 3. Contextos de navegador aislados
```python
@pytest.fixture
def isolated_context(browser):
    """Crea un contexto de navegador aislado."""
    context = browser.new_context()
    yield context
    context.close()
```

## 3. ¿Cuándo usar E2E vs Integración?

### Usar E2E cuando:
- Necesitas validar el flujo completo del usuario
- Quieres probar integraciones críticas entre sistemas
- La lógica de negocio está distribuida en múltiples capas
- Necesitas validar la experiencia del usuario real
- Quieres detectar problemas de configuración/entorno

### Usar Integración cuando:
- Solo necesitas probar la interacción entre componentes
- El flujo de usuario es simple
- Las pruebas E2E son demasiado lentas
- Puedes mockear dependencias externas
- Quieres pruebas rápidas en el ciclo de desarrollo

### Usar Unitarias cuando:
- Necesitas probar lógica de negocio aislada
- Quieres pruebas extremadamente rápidas
- El comportamiento es puramente algorítmico
- Puedes probar todos los casos edge fácilmente

## 4. Pirámide de pruebas

```
        /\
       /E2E\          10% - Lentas, costosas, frágiles
      /------\
     /Integración\    30% - Medianas, balanceadas
    /--------------\
   /  Unitarias      \   60% - Rápidas, confiables
  /------------------\
```

La regla general: **más pruebas unitarias, menos E2E**.

## 5. Antipatrones comunes en E2E

### ❌ Antipatrón 1: Demasiadas pruebas E2E
```python
# MAL: Probar cada campo con E2E
def test_field_validation_1(page): ...
def test_field_validation_2(page): ...
def test_field_validation_3(page): ...
# 100 tests de validación de campos

# BIEN: Validación de campos con unitarias/integración
# Solo E2E para flujos críticos de usuario
```

### ❌ Antipatrón 2: Locators frágiles
```python
# MAL: XPath que se rompe con cualquier cambio
page.locator('xpath=/html/body/div[1]/form/input[1]')

# BIEN: data-testid estable
page.locator('[data-testid="task-title-input"]')
```

### ❌ Antipatrón 3: Esperas fijas
```python
# MAL: time.sleep es la raíz de todos los males
time.sleep(5)

# BIEN: Esperas inteligentes de Playwright
expect(element).to_be_visible()
```

### ❌ Antipatrón 4: Tests que dependen del orden
```python
# MAL: Test 2 depende de Test 1
def test_1_create(page):
    page.fill('[data-testid="task-title-input"]', 'Task')
    page.click('[data-testid="create-task-button"]')

def test_2_delete(page):
    # Asume que Test 1 creó la tarea
    page.click('[data-testid="delete-button"]')

# BIEN: Cada test es independiente
```

## 6. Costo de las pruebas E2E

| Aspecto | Unitarias | Integración | E2E |
|---------|-----------|-------------|-----|
| Velocidad | Segundos | Minutos | Horas |
| Costo de mantenimiento | Bajo | Medio | Alto |
| Fragilidad | Baja | Media | Alta |
| Cobertura de flujos | Baja | Media | Alta |
| Detección de bugs | Lógica | Integración | Sistémicos |

## 7. Reflexión

Responde las siguientes preguntas:

1. **¿Qué son los flaky tests y por qué son especialmente comunes en E2E?**
   - Son pruebas que pasan/fallan intermitentemente sin cambios en el código
   - Comunes en E2E por: timing issues, dependencia de estado, recursos externos, concurrencia

2. **¿Cómo garantizarías el aislamiento entre tests en una suite E2E?**
   - Limpieza antes/después de cada test (autouse fixtures)
   - Bases de datos en memoria o archivos únicos por test
   - Contextos de navegador aislados
   - Cada test crea su propio estado

3. **¿En qué casos usarías E2E en lugar de pruebas de integración?**
   - Flujos críticos de usuario completos
   - Integraciones entre múltiples sistemas
   - Validación de experiencia de usuario real
   - Detección de problemas de configuración
   - Cuando la lógica está distribuida en múltiples capas

---

## Conclusión

Las pruebas E2E son necesarias pero deben usarse con moderación. La clave es:
- **Menos es más**: Solo E2E para flujos críticos
- **Aislamiento**: Cada test independiente
- **Locators robustos**: data-testid sobre XPath
- **Esperas inteligentes**: Nunca time.sleep
- **Pirámide de pruebas**: Más unitarias, menos E2E